# CrewAI + LangGraph Integration: Restaurant Social Media Marketing Pipeline

## Scenario

This builds on the pure-CrewAI restaurant marketing project. Instead of running the
5-agent CrewAI crew standalone, we wrap it inside a **LangGraph graph** as one node,
then add a second LangGraph node that post-processes the crew's output into two
separate, clean deliverables: the blog post and the social media content.

**Important:** there is no `main.py` and no `crewai run` anywhere in this notebook.
CrewAI is used purely as a Python library — `Crew(...).kickoff(...)` is just a
function call. LangGraph treats the entire CrewAI crew as a single "leaf" node: it
has no visibility into the five internal CrewAI agents, their tools, or their
context-chaining. From LangGraph's perspective, one node goes in, one result comes
back out.

### Graph shape

```
        +----------------------+       +--------------------------+
START ->| run_marketing_crew   | ----> | split_deliverables        | -> END
        | (CrewAI crew inside) |       | (parses blog vs. social)  |
        +----------------------+       +--------------------------+
```

- **`run_marketing_crew`** — calls `crew.kickoff(inputs=...)`, using the LangGraph
  state to parameterize the crew (e.g. which dish category to focus on), and stores
  the raw crew result back into state.
- **`split_deliverables`** — takes the raw crew result (a single blob of text
  containing both the blog post and the social posts) and splits it into two clean
  state fields: `blog_post` and `social_media_content`.

This mirrors a common real pattern: CrewAI is great at orchestrating a *sequential
content pipeline* internally, while LangGraph is used at a higher level to manage
*application state* and decide what happens with the result (save to a DB, send to
a review node, trigger a human-in-the-loop approval step, etc.).

## Assignment Notebook (fill in the TODOs)

## Section 1 — Environment Setup

In [ ]:
import os
import sqlite3
from typing import TypedDict, Optional

from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

# TODO 1: Import what you need from langgraph.graph to build a graph
# (StateGraph, END).

# TODO 2: Make sure your OPENAI_API_KEY (or other LLM provider key) is set in the
# environment before running this notebook.


## Section 2 — Mock Database and Tools (same as the pure-CrewAI project)

We reuse the exact same `restaurant_marketing.db` setup and tools from the
pure-CrewAI assignment: `dishes`, `nutrition`, and `produce` tables, each backed by
a CrewAI `@tool`.


In [ ]:
DB_PATH = "restaurant_marketing.db"


def setup_database():
    """Create and populate the mock restaurant marketing database."""
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # TODO 3: Create the `dishes` table.
    # Columns: id (INTEGER PRIMARY KEY), name, cuisine, main_ingredients, description
    # NOTE: this time `cuisine` should be either "Indian Fusion" or "Pakistani Fusion"
    # -- you will filter on this value later via {focus_cuisine}.
    cursor.execute("""
        CREATE TABLE dishes (
            -- your columns here
        )
    """)

    # TODO 4: Create the `nutrition` table.
    # Columns: dish_id, calories, protein_g, carbs_g, fat_g, fiber_g, notes
    cursor.execute("""
        CREATE TABLE nutrition (
            -- your columns here
        )
    """)

    # TODO 5: Create the `produce` table.
    # Columns: ingredient, region, season, local_farm_source
    cursor.execute("""
        CREATE TABLE produce (
            -- your columns here
        )
    """)

    # TODO 6: Insert at least 6 dishes, with roughly half tagged "Indian Fusion"
    # and half "Pakistani Fusion" so {focus_cuisine} filtering has something to do.
    dishes = [
        # (1, "Dish Name", "Indian Fusion", "ingredients...", "description..."),
    ]
    cursor.executemany("INSERT INTO dishes VALUES (?, ?, ?, ?, ?)", dishes)

    # TODO 7: Insert matching nutrition rows for each dish_id.
    nutrition = [
        # (1, 420, 28.0, 32.0, 19.0, 3.0, "note"),
    ]
    cursor.executemany("INSERT INTO nutrition VALUES (?, ?, ?, ?, ?, ?, ?)", nutrition)

    # TODO 8: Insert at least 6 produce/sourcing rows.
    produce = [
        # ("Tomato", "Local Valley Farms", "Summer", "Green Acres Co-op"),
    ]
    cursor.executemany("INSERT INTO produce VALUES (?, ?, ?, ?)", produce)

    conn.commit()
    conn.close()
    print("Database setup complete:", DB_PATH)


setup_database()


In [ ]:
# TODO 9: Implement get_trending_dishes so it optionally filters by cuisine.
# If `cuisine_filter` is a non-empty string, only return dishes whose `cuisine`
# column matches it (case-insensitive, partial match is fine). Otherwise return
# all dishes.
@tool("Trending Dishes Lookup")
def get_trending_dishes(cuisine_filter: str = "") -> str:
    """Returns the restaurant's new/trending fusion dishes. If cuisine_filter is
    provided (e.g. 'Indian Fusion' or 'Pakistani Fusion'), only dishes matching
    that cuisine are returned; otherwise all dishes are returned."""
    # your implementation here
    pass


# TODO 10: Implement get_nutrition_info (same as the pure-CrewAI version --
# join dishes + nutrition, case-insensitive partial match on dish_name).
@tool("Nutrition Info Lookup")
def get_nutrition_info(dish_name: str) -> str:
    """Given a dish name (or partial name), returns its nutrition facts:
    calories, protein, carbs, fat, and fiber."""
    # your implementation here
    pass


# TODO 11: Implement get_local_produce_info (same as the pure-CrewAI version).
@tool("Local Produce Sourcing Lookup")
def get_local_produce_info() -> str:
    """Returns sourcing details for key ingredients: region, season, and
    local farm/supplier."""
    # your implementation here
    pass


## Section 3 — Agents (same 5-agent pipeline)

In [ ]:
# TODO 12: Define all 5 agents. This is identical to the pure-CrewAI assignment --
# copy your working agent definitions over, or redo them here.
# - trend_researcher: tools=[get_trending_dishes]
# - nutrition_analyst: tools=[get_nutrition_info]
# - sourcing_specialist: tools=[get_local_produce_info]
# - blog_writer: no tools
# - social_media_strategist: no tools
trend_researcher = Agent(role="", goal="", backstory="", tools=[], verbose=True)
nutrition_analyst = Agent(role="", goal="", backstory="", tools=[], verbose=True)
sourcing_specialist = Agent(role="", goal="", backstory="", tools=[], verbose=True)
blog_writer = Agent(role="", goal="", backstory="", verbose=True)
social_media_strategist = Agent(role="", goal="", backstory="", verbose=True)


## Section 4 — Tasks

One addition versus the pure-CrewAI version: `research_task`'s description now
references `{focus_cuisine}`, a placeholder that gets filled in at runtime from
`crew.kickoff(inputs={...})`. This is how the LangGraph state flows *into* the crew.


In [ ]:
# TODO 13: Define research_task.
# - description should reference {focus_cuisine} so it can be filled in at
#   kickoff() time, and should instruct the agent to call the trending dishes
#   tool with cuisine_filter='{focus_cuisine}'.
# - expected_output: a list of matching dishes with a short hook each.
research_task = Task(
    description="",
    expected_output="",
    agent=trend_researcher,
)

# TODO 14: Define nutrition_task. Same as the pure-CrewAI version.
# - context: [research_task]
nutrition_task = Task(
    description="",
    expected_output="",
    agent=nutrition_analyst,
    context=[],
)

# TODO 15: Define sourcing_task. Same as the pure-CrewAI version.
# - context: [research_task]
sourcing_task = Task(
    description="",
    expected_output="",
    agent=sourcing_specialist,
    context=[],
)

# TODO 16: Define blog_task.
# IMPORTANT: instruct the writer to put the blog post under the exact Markdown
# heading '## Blog Post' -- the LangGraph post-processing node in Section 8
# depends on this exact heading to split the output.
# - context: [research_task, nutrition_task, sourcing_task]
blog_task = Task(
    description="",
    expected_output="",
    agent=blog_writer,
    context=[],
)

# TODO 17: Define social_media_task.
# IMPORTANT: instruct the strategist to use the exact headings
# '## Facebook Post' and '## Twitter Thread' -- again, required for the
# LangGraph post-processing node to parse the output correctly.
# - context: [blog_task]
social_media_task = Task(
    description="",
    expected_output="",
    agent=social_media_strategist,
    context=[],
)


## Section 5 — Assemble the Crew (no `main.py`, no `crewai run`)

In [ ]:
# TODO 18: Assemble the Crew (agents + tasks in matching order, Process.sequential).
# Do NOT call kickoff() here -- that happens inside the LangGraph node below,
# so the crew can be parameterized by LangGraph state.
crew = Crew(
    agents=[],
    tasks=[],
    process=None,
    verbose=True,
)


## Section 6 — Define the LangGraph State

The state is the single shared dict that flows between LangGraph nodes. It needs to
carry whatever comes in from the caller (`focus_cuisine`), the raw output from the
CrewAI crew (`raw_crew_output`), and the **three** split-out deliverables produced by
the second node: `blog_post`, `facebook_post`, and `twitter_thread`.

Note this is three separate fields, not two. An earlier version of this node lumped
Facebook and Twitter into a single `social_media_content` field by taking
"everything from the Facebook marker to the end of the string" — which meant that if
the Twitter section came out mangled, missing, or the marker didn't match exactly,
it silently vanished into (or was missing from) that combined blob with no way to
tell which one failed. Splitting into three independently-located markers fixes
that: each section is found on its own, so a missing/malformed Twitter section
shows up as an empty `twitter_thread` field you can actually detect and handle,
rather than disappearing inside a bigger string.


In [ ]:
# TODO 19: Define the LangGraph state as a TypedDict with these fields:
# - focus_cuisine: str              (input -- which cuisine to focus the crew on)
# - raw_crew_output: Optional[str]  (set by node 1, the full crew.kickoff() result)
# - blog_post: Optional[str]        (set by node 2, just the blog post text)
# - facebook_post: Optional[str]    (set by node 2, just the Facebook post text)
# - twitter_thread: Optional[str]   (set by node 2, just the Twitter/X thread text)
class MarketingState(TypedDict):
    pass


## Section 7 — Node 1: `run_marketing_crew`

This node's entire job is: take relevant fields out of LangGraph state, pass them
into `crew.kickoff(inputs=...)`, and write the crew's result back into state. It
does NOT know or care that internally this triggers 5 sequential CrewAI agents.


In [ ]:
# TODO 20: Implement run_marketing_crew.
# - Read state["focus_cuisine"].
# - Call crew.kickoff(inputs={"focus_cuisine": ...}) to pass it into the crew.
# - Return a dict with key "raw_crew_output" set to str(result).
def run_marketing_crew(state: MarketingState) -> dict:
    """LangGraph node: runs the entire CrewAI crew and stores its raw output."""
    # your implementation here
    pass


## Section 8 — Node 2: `split_deliverables`

This node has no CrewAI involvement at all — it's a plain LangGraph node that
post-processes `raw_crew_output` into three clean fields: `blog_post`,
`facebook_post`, and `twitter_thread`. This is the kind of task LangGraph is good at
that CrewAI's `Process.sequential` doesn't give you directly: free-form control flow
over the crew's result.

Each marker (`## Blog Post`, `## Facebook Post`, `## Twitter Thread`) is located
**independently** with `str.find()`, then the markers actually found are sorted by
position and each section runs from its own marker to the start of the next one
found (or to the end of the string for the last one). This way a missing or
misspelled marker only empties out that one field instead of corrupting or merging
adjacent sections.


In [ ]:
# TODO 21: Implement split_deliverables.
# - Read state["raw_crew_output"].
# - Locate each of the three markers independently with str.find():
#     "## Blog Post", "## Facebook Post", "## Twitter Thread"
# - Keep only the markers that were actually found (index != -1), and sort
#   them by their position in the text.
# - For each found marker, its section runs from its own position up to the
#   position of the NEXT found marker (not a fixed next-in-list marker --
#   a marker that wasn't found shouldn't create a gap). The last found
#   marker's section runs to the end of the string.
# - If NO markers are found at all, fall back to putting the entire raw
#   string into blog_post and leaving facebook_post / twitter_thread as "".
# - Return a dict with keys: blog_post, facebook_post, twitter_thread.
def split_deliverables(state: MarketingState) -> dict:
    """LangGraph node: splits the crew's raw output into blog_post,
    facebook_post, and twitter_thread using the '## Blog Post' /
    '## Facebook Post' / '## Twitter Thread' markers the tasks were
    instructed to produce."""
    # your implementation here
    pass


## Section 9 — Build and Compile the Graph

In [ ]:
# TODO 22: Build the graph.
# - Instantiate StateGraph(MarketingState).
# - Add both nodes: "run_marketing_crew" and "split_deliverables".
# - Set the entry point to "run_marketing_crew".
# - Add an edge from "run_marketing_crew" to "split_deliverables".
# - Add an edge from "split_deliverables" to END.
# - Compile the graph into `app`.
graph = None
app = None


## Section 10 — Invoke the Graph

In [ ]:
# TODO 23: Invoke the compiled graph with an initial state:
# {"focus_cuisine": "Pakistani Fusion"} (or "Indian Fusion" -- try both!)
# Store the result in `final_state`, then print out blog_post, facebook_post,
# and twitter_thread separately. Also print a warning if twitter_thread comes
# back empty, so a formatting failure is visible instead of silent.
final_state = None


## Reflection Questions

1. Why does `run_marketing_crew` pass `inputs={"focus_cuisine": ...}` into
   `kickoff()` rather than, say, mutating the task descriptions directly at
   runtime?
2. `split_deliverables` uses a simple string-marker split (looking for a heading
   like `## Facebook Post`). What's a more robust way to get structured output out
   of the crew's final task, and which CrewAI feature would you reach for
   (hint: look at `Task(output_pydantic=...)` or `output_json=...`)?
3. If you wanted a **human-in-the-loop approval step** between the blog post being
   drafted and the social posts being generated, would you add that as a CrewAI
   task, or as a separate LangGraph node? Justify your answer in terms of which
   framework owns which kind of control flow.
4. This graph only has one path: `run_marketing_crew -> split_deliverables -> END`.
   Sketch (in words) how you'd add a conditional edge that routes to a
   `regenerate_social_posts` node if `social_media_content` comes back empty.
